# Retail Data ELT Lab — SQLite and Python

**Student starter notebook**

Use this notebook with the lab instructions to load the instructor-provided CSV files into SQLite, inspect their schemas, create staging views, run data-quality checks, and answer business questions.

**Tools:** Python, pandas, SQLite, Google Colab. No BigQuery or dbt setup is required.

Before starting, put `customers.csv`, `products.csv`, `orders_raw.csv`, and `orders_clean.csv` in a folder in your Google Drive, or use the direct-upload option in Step 1. Run cells from top to bottom. Use actual column names from your files; examples below are templates.

In [24]:
import sqlite3
from pathlib import Path
import pandas as pd

print("Python and SQLite are ready.")
print("SQLite version:", sqlite3.sqlite_version)
print("pandas version:", pd.__version__)

Python and SQLite are ready.
,SQLite version: 3.45.1
,pandas version: 2.2.3


## 1. Make the CSV files available

Choose **one** option: Google Drive (recommended) or direct upload. If using Drive, copy the four CSV files into a folder you own, for example `MyDrive/Retail_ELT_Lab`.

In [25]:
# OPTION A (recommended): Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/Retail_ELT_Lab')  # Change if your folder differs

required_files = [
    'customers.csv',
    'products.csv',
    'orders_raw.csv',
    'orders_clean.csv'
]

for filename in required_files:
    file_path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
,customers.csv: NOT FOUND
,products.csv: NOT FOUND
,orders_raw.csv: NOT FOUND
,orders_clean.csv: NOT FOUND


In [26]:
# OPTION B: Upload files directly from your computer.
# Run this cell ONLY if you are not using Google Drive.
from google.colab import files
uploaded = files.upload()
DATA_DIR = Path('/content')
#
# After upload, verify that all four CSV files are in /content.

Saving customers.csv to customers (1).csv
,Saving orders_clean.csv to orders_clean (1).csv
,Saving orders_raw.csv to orders_raw (1).csv
,Saving products.csv to products (1).csv


## 2. Read and inspect the CSV files

Run this after all four files show as found (or after direct upload). Review the columns and sample records before writing SQL.

In [27]:
customers_df = pd.read_csv(DATA_DIR / 'customers.csv')
products_df = pd.read_csv(DATA_DIR / 'products.csv')
orders_raw_df = pd.read_csv(DATA_DIR / 'orders_raw.csv')
orders_clean_df = pd.read_csv(DATA_DIR / 'orders_clean.csv')

dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df,
    'orders_clean': orders_clean_df
}

for name, df in dataframes.items():
    print(f"\n--- {name} ---")
    print("Shape (rows, columns):", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


,--- customers ---
,Shape (rows, columns): (500, 7)
,Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'state', 'signup_date']


,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24



,--- products ---
,Shape (rows, columns): (20, 4)
,Columns: ['product_id', 'product_name', 'category', 'price']


,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99



,--- orders_raw ---
,Shape (rows, columns): (5100, 6)
,Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped



,--- orders_clean ---
,Shape (rows, columns): (5000, 6)
,Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O00001,C0397,P009,4,2025-09-18,delivered
1,O00002,C0147,P016,2,2024-03-16,delivered
2,O00003,C0232,P006,4,2025-11-29,cancelled
3,O00004,C0496,P011,3,2025-07-17,shipped
4,O00005,C0278,P010,3,2025-12-29,pending


In [28]:
for name, df in dataframes.items():
    print(f"\n--- {name}: data types ---")
    display(df.dtypes.to_frame('data_type'))
    print("Missing values by column:")
    display(df.isna().sum().to_frame('missing_count'))


,--- customers: data types ---


,data_type
customer_id,object
first_name,object
last_name,object
email,object
city,object
state,object
signup_date,object


Missing values by column:


,missing_count
customer_id,0
first_name,0
last_name,0
email,0
city,0
state,0
signup_date,0



,--- products: data types ---


,data_type
product_id,object
product_name,object
category,object
price,float64


... (6 outputs left out)


,--- orders_clean: data types ---


,data_type
order_id,object
customer_id,object
product_id,object
quantity,int64
order_date,object
status,object


Missing values by column:


,missing_count
order_id,0
customer_id,0
product_id,0
quantity,0
order_date,0
status,0


## 3. Create a SQLite database and load the raw tables

Load `customers.csv`, `products.csv`, and `orders_raw.csv` into SQLite. Keep `orders_clean.csv` as a reference file for later comparison; do not use it to replace the raw orders table.

In [29]:
DB_PATH = Path('/content/retail_lab.db')
conn = sqlite3.connect(DB_PATH)

raw_dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df
}

for table_name, df in raw_dataframes.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {len(df):,} rows into {table_name}")

print("SQLite database:", DB_PATH)

Loaded 500 rows into customers
,Loaded 20 rows into products
,Loaded 5,100 rows into orders_raw
,SQLite database: /content/retail_lab.db


In [30]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)
display(tables)

for table_name in ['customers', 'products', 'orders_raw']:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS row_count FROM "{table_name}"',
        conn
    ).iloc[0]['row_count']
    print(f"{table_name}: {count:,} rows")

,name
0,customers
1,orders_raw
2,products


customers: 500 rows
,products: 20 rows
,orders_raw: 5,100 rows


In [31]:
display(pd.read_sql_query('SELECT * FROM orders_raw LIMIT 10;', conn))

,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped
5,O03546,C0343,P007,5,2024-01-27,delivered
6,O02403,C0189,P020,4,2025-07-28,pending
7,O04813,C0455,P015,3,2024-11-04,shipped
8,O01179,C0424,P010,3,2025-12-05,delivered
9,O00202,C0103,P019,5,2025-07-17,delivered


## 4. Inspect SQLite schemas and table grain

Review the exact field names and data types. Decide what one row represents in each table (for example, one customer, one product, one order, or one order line).

In [32]:
for table_name in ['customers', 'products', 'orders_raw']:
    print(f"\n--- {table_name} schema ---")
    display(pd.read_sql_query(f'PRAGMA table_info("{table_name}");', conn))


,--- customers schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,email,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,state,TEXT,0,None,0
6,6,signup_date,TEXT,0,None,0



,--- products schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,TEXT,0,None,0
1,1,product_name,TEXT,0,None,0
2,2,category,TEXT,0,None,0
3,3,price,REAL,0,None,0



,--- orders_raw schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,order_date,TEXT,0,None,0
5,5,status,TEXT,0,None,0


## 5. Create staging views with SQL

These starter views select all columns so they can run before you know the final field names. Improve them by selecting fields needed for analysis, standardizing text where appropriate, and documenting quality rules. If you edit a view definition, rerun its cell; each cell drops and recreates the view.

In [33]:
conn.execute("DROP VIEW IF EXISTS stg_customers;")
conn.execute("""
CREATE VIEW stg_customers AS
SELECT *
FROM customers;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_customers LIMIT 5;', conn))

,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24


In [34]:
conn.execute("DROP VIEW IF EXISTS stg_products;")
conn.execute("""
CREATE VIEW stg_products AS
SELECT *
FROM products;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_products LIMIT 5;', conn))

,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99


In [35]:
conn.execute("DROP VIEW IF EXISTS stg_orders;")
conn.execute("""
CREATE VIEW stg_orders AS
SELECT *
FROM orders_raw;
""")
conn.commit()
display(pd.read_sql_query('SELECT * FROM stg_orders LIMIT 5;', conn))

,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped


### Example transformation pattern — adapt to your schema

This is only an example; it will not run unless the fields exist in your table.

```sql
SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    CAST(quantity AS REAL) * CAST(unit_price AS REAL) AS line_total
FROM orders_raw;
```

SQLite casts can behave unexpectedly for invalid text values. Inspect the raw values and document conversion rules before relying on calculated values.

## 6. Data-quality checks

The examples below assume `customer_id` exists and that the customers table should have one row per customer. Change the field name or rule to match your actual schema.

In [36]:
# Example: missing or blank customer IDs.
# Uncomment and edit customer_id if that is the actual identifier column.

display(pd.read_sql_query("""
SELECT COUNT(*) AS missing_customer_ids
FROM stg_customers
WHERE customer_id IS NULL
   OR TRIM(CAST(customer_id AS TEXT)) = '';
""", conn))

,missing_customer_ids
0,0


In [37]:
# Example: duplicate customer IDs (only if one row per customer is expected).
#
display(pd.read_sql_query("""
SELECT customer_id, COUNT(*) AS records
FROM stg_customers
GROUP BY customer_id
HAVING COUNT(*) > 1;
""", conn))

,customer_id,records


## 7. Analytical SQL — answer business questions

Use your actual column names. First inspect `stg_orders`, `stg_customers`, and `stg_products` and identify:
- the order identifier and customer identifier
- the product identifier (if each row represents an order line)
- the order date
- the quantity, unit price, discount, and line total (where available)
- any order status, region, or category fields

**Important:** The queries below are templates, not guaranteed to run unchanged. Replace placeholder column names with fields in your files. If a field is not available, explain that limitation and choose a suitable alternative. Run at least four queries and write 1–3 sentences interpreting each result.


In [38]:
# Query A — overall sales and order-line summary
# Adapt the placeholders to your schema. If line_total is absent, calculate it
# from quantity * unit_price (and apply discounts only if their meaning is clear).

display(pd.read_sql_query("""
SELECT
    COUNT(*) AS order_line_count,
    COUNT(DISTINCT order_id) AS distinct_orders,
    SUM(quantity) AS units_sold,
    ROUND(SUM(o.quantity * p.price), 2) AS total_sales,
    ROUND(AVG(o.quantity * p.price), 2) AS average_line_value
FROM stg_orders AS o
JOIN stg_products AS p
    ON o.product_id = p.product_id;
""", conn))

# Interpretation prompts:
# 1. How many rows are present compared with distinct orders?
# 2. Why might these counts be different?
# 3. What does the average line value tell you (and what does it not tell you)?


,order_line_count,distinct_orders,units_sold,total_sales,average_line_value
0,5100,5000,14947,1113264.53,218.29


In [39]:
# Query B — Product Performance
display(pd.read_sql_query("""
SELECT
    p.product_name,
    SUM(o.quantity) AS units_sold,
    ROUND(SUM(o.quantity * p.price), 2) AS product_sales,
    COUNT(DISTINCT o.order_id) AS orders_including_product
FROM stg_orders AS o
JOIN stg_products AS p
    ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY product_sales DESC
LIMIT 10;
""", conn))


# Query C — Customer-level Performance
display(pd.read_sql_query("""
SELECT
    c.customer_id,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(o.quantity * p.price), 2) AS customer_sales,
    ROUND(AVG(o.quantity * p.price), 2) AS average_line_value
FROM stg_customers AS c
LEFT JOIN stg_orders AS o
    ON c.customer_id = o.customer_id
LEFT JOIN stg_products AS p
    ON o.product_id = p.product_id
GROUP BY c.customer_id
ORDER BY customer_sales DESC;
""", conn))


# Query D — Monthly Sales Trend
display(pd.read_sql_query("""
SELECT
    substr(o.order_date, 1, 7) AS order_month,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(o.quantity * p.price), 2) AS monthly_sales
FROM stg_orders AS o
JOIN stg_products AS p
    ON o.product_id = p.product_id
WHERE o.order_date IS NOT NULL
GROUP BY substr(o.order_date, 1, 7)
ORDER BY order_month;
""", conn))


# Query E — Duplicate Order Investigation
display(pd.read_sql_query("""
SELECT
    order_id,
    COUNT(*) AS rows_per_order
FROM stg_orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY rows_per_order DESC
LIMIT 20;
""", conn))

,product_name,units_sold,product_sales,orders_including_product
0,Smart Watch,926,185190.74,298
1,Cookware Set,648,103673.52,225
2,Winter Jacket,686,102893.14,229
3,Running Shoes,746,96972.54,255
4,Mechanical Keyboard,732,87832.68,251
5,Tennis Racket,804,80391.96,280
6,Wireless Headphones,760,68392.40,252
7,Coffee Maker,854,68311.46,282
8,Dumbbell Pair,739,51722.61,261
9,Denim Jeans,748,44872.52,238


,customer_id,order_count,customer_sales,average_line_value
0,C0152,13,6777.42,451.83
1,C0103,13,5328.54,380.61
2,C0232,18,5269.37,292.74
3,C0301,17,5219.44,307.03
4,C0192,14,4899.47,349.96
...,...,...,...,...
495,C0481,4,374.90,93.73
496,C0136,3,334.89,111.63
497,C0389,3,269.94,89.98
498,C0427,2,269.93,134.97


,order_month,order_count,monthly_sales
0,2024-01,210,52877.46
1,2024-02,197,41020.31
2,2024-03,196,46057.72
3,2024-04,212,43472.88
4,2024-05,215,46684.57
5,2024-06,203,47876.06
6,2024-07,204,48946.04
7,2024-08,199,44149.94
8,2024-09,211,44756.46
9,2024-10,203,39988.17


,order_id,rows_per_order
0,O04994,2
1,O04974,2
2,O04949,2
3,O04864,2
4,O04771,2
5,O04770,2
6,O04691,2
7,O04673,2
8,O04666,2
9,O04618,2


## 8. Reconcile your results with `orders_clean.csv`

Treat `orders_clean.csv` as a reference dataset. Do not assume it has the same grain as `orders_raw` or your staging view. First inspect the fields and determine whether each row represents an order or an order line.

Complete these checks:
1. Compare row counts and column names.
2. Check for missing and duplicate business keys in the reference file.
3. Compare totals only when the two datasets use the same grain and the same definition of sales.
4. If both datasets share a reliable key, compare key coverage and identify records present in only one dataset.
5. If you compare numeric values, report the absolute difference and use a sensible rounding tolerance.
6. Explain at least two differences or data-quality issues you found. If the files cannot be compared directly, explain why and define the transformation needed to make them comparable.


In [40]:
# Basic profile of the reference dataset
print("Reference row count:", len(orders_clean_df))
print("Reference columns:", orders_clean_df.columns.tolist())

print("\nMissing values by column:")
display(orders_clean_df.isna().sum().rename("missing_count").to_frame())

print("\nDuplicate full rows:", int(orders_clean_df.duplicated().sum()))
display(orders_clean_df.head())


# TODO 1: Identify the business key (for example, order_id or order_line_id)
# in the reference data and check whether it is missing or duplicated.
#
# Example template — replace order_id with the correct key:

# TODO 1: Check business key
reference_key = "order_id"

print("\nMissing reference keys:",
      int(orders_clean_df[reference_key].isna().sum()))

print("Duplicate reference keys:",
      int(orders_clean_df[reference_key].duplicated().sum()))


# TODO 2: Create a comparable result from your staging data.
# Example template — adapt fields and the aggregation level:
#
# transformed_df = pd.read_sql_query("""
# SELECT order_id, ROUND(SUM(line_total), 2) AS calculated_sales
# FROM stg_orders
# GROUP BY order_id
# """, conn)

# Since the reference dataset does not contain sales_amount,
# we first compare the raw and reference dataset structures and keys.

print("\n--- Raw Data Checks ---")
print("Raw rows:", len(orders_raw_df))

print("Missing raw order IDs:",
      int(orders_raw_df["order_id"].isna().sum()))

print("Distinct raw orders:",
      orders_raw_df["order_id"].nunique())


# TODO 3: If both datasets contain the same key, compare key coverage.
# Rename the reference key and value columns to match the transformed result first.
#
# comparison = transformed_df.merge(
#     orders_clean_df[[reference_key, "sales_amount"]].rename(
#         columns={reference_key: "order_id", "sales_amount": "reference_sales"}
#     ),
#     on="order_id",
#     how="outer",
#     indicator=True
# )
# print(comparison["_merge"].value_counts())

# Compare order IDs between raw and reference datasets

raw_ids = set(orders_raw_df["order_id"].dropna())
clean_ids = set(orders_clean_df["order_id"].dropna())

print("\n--- Order ID Coverage ---")
print("Orders only in raw:", len(raw_ids - clean_ids))
print("Orders only in clean:", len(clean_ids - raw_ids))
print("Orders present in both:", len(raw_ids & clean_ids))


# TODO 4: For records matched in both datasets, calculate a numeric difference.
# matched = comparison[comparison["_merge"] == "both"].copy()
# matched["absolute_difference"] = (
#     matched["calculated_sales"] - matched["reference_sales"]
# ).abs()
# print("Rows differing by more than 0.01:",
#       int((matched["absolute_difference"] > 0.01).sum()))
# display(matched.sort_values("absolute_difference", ascending=False).head(10))

# Check repeated order IDs in raw data

raw_order_counts = orders_raw_df["order_id"].value_counts()

print("\n--- Raw Order Grain Check ---")
print("Order IDs appearing more than once:",
      int((raw_order_counts > 1).sum()))

display(raw_order_counts[raw_order_counts > 1].head(10))


# Note: The templates remain commented because the CSV column names and grain
# must be confirmed before choosing keys or comparing totals.
#
# The reference dataset has no sales_amount column, so a numeric
# sales reconciliation cannot be performed directly using the template.
# Repeated order IDs may represent multiple order lines, not duplicate orders.

Reference row count: 5000
,Reference columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']
,
,Missing values by column:


,missing_count
order_id,0
customer_id,0
product_id,0
quantity,0
order_date,0
status,0



,Duplicate full rows: 0


,order_id,customer_id,product_id,quantity,order_date,status
0,O00001,C0397,P009,4,2025-09-18,delivered
1,O00002,C0147,P016,2,2024-03-16,delivered
2,O00003,C0232,P006,4,2025-11-29,cancelled
3,O00004,C0496,P011,3,2025-07-17,shipped
4,O00005,C0278,P010,3,2025-12-29,pending



,Missing reference keys: 0
,Duplicate reference keys: 0
,
,--- Raw Data Checks ---
,Raw rows: 5100
,Missing raw order IDs: 0
,Distinct raw orders: 5000
,
,--- Order ID Coverage ---
,Orders only in raw: 0
,Orders only in clean: 0
,Orders present in both: 5000
,
,--- Raw Order Grain Check ---
,Order IDs appearing more than once: 100


,count
order_id,
O02616,2
O03937,2
O03107,2
O03240,2
O00900,2
O01874,2
O01154,2
O01975,2
O04598,2


***Reconciliation Findings***

The reference dataset contains 5,000 rows with no missing values, duplicate full rows, or duplicate order IDs. The raw dataset contains 5,100 rows and 5,000 distinct order IDs.

The comparison shows that all 5,000 order IDs are present in both datasets. There are no orders exclusive to either dataset, indicating 100% order ID coverage.

The raw dataset has 100 order IDs appearing more than once. These repeated IDs may represent multiple order lines belonging to the same order rather than duplicate records. Further inspection of the product IDs and quantities is needed to confirm this.

**Key observation:** The raw dataset contains 100 more rows than the reference dataset, even though both contain the same 5,000 unique orders.

In [41]:
# Inspect repeated order IDs
repeated_ids = raw_order_counts[raw_order_counts > 1].index

repeated_orders = orders_raw_df[
    orders_raw_df["order_id"].isin(repeated_ids)
].sort_values("order_id")

display(repeated_orders.head(20))

,order_id,customer_id,product_id,quantity,order_date,status
678,O00021,C0369,P020,3,2025-07-31,shipped
2924,O00021,C0369,P020,3,2025-07-31,shipped
5015,O00022,C0022,P001,1,2025-10-18,delivered
4784,O00022,C0022,P001,1,2025-10-18,delivered
204,O00154,C0234,P012,1,2025-11-13,delivered
4182,O00154,C0234,P012,1,2025-11-13,delivered
4353,O00184,C0053,P005,1,2025-04-04,delivered
4921,O00184,C0053,P005,1,2025-04-04,delivered
826,O00301,C0380,P006,1,2024-05-20,delivered
2020,O00301,C0380,P006,1,2024-05-20,delivered


In [42]:
# Check exact duplicate rows in raw dataset
print("Exact duplicate rows in raw data:",
      int(orders_raw_df.duplicated().sum()))

# Check missing values in raw dataset
print("\nMissing values in raw data:")
display(orders_raw_df.isna().sum().to_frame("missing_count"))

# Check duplicate order lines
duplicate_lines = orders_raw_df[
    orders_raw_df.duplicated(keep=False)
].sort_values("order_id")

print("\nSample exact duplicate records:")
display(duplicate_lines.head(10))

Exact duplicate rows in raw data: 100
,
,Missing values in raw data:


,missing_count
order_id,0
customer_id,173
product_id,0
quantity,0
order_date,0
status,0



,Sample exact duplicate records:


,order_id,customer_id,product_id,quantity,order_date,status
678,O00021,C0369,P020,3,2025-07-31,shipped
2924,O00021,C0369,P020,3,2025-07-31,shipped
5015,O00022,C0022,P001,1,2025-10-18,delivered
4784,O00022,C0022,P001,1,2025-10-18,delivered
204,O00154,C0234,P012,1,2025-11-13,delivered
4182,O00154,C0234,P012,1,2025-11-13,delivered
4353,O00184,C0053,P005,1,2025-04-04,delivered
4921,O00184,C0053,P005,1,2025-04-04,delivered
826,O00301,C0380,P006,1,2024-05-20,delivered
2020,O00301,C0380,P006,1,2024-05-20,delivered


***Reconciliation Findings and Data Quality Anomalies***

The reference dataset contains 5,000 rows with no missing values, duplicate full rows, or duplicate order IDs.

The raw dataset contains 5,100 rows and 5,000 distinct order IDs. Comparing the raw and reference datasets shows that all 5,000 order IDs are present in both datasets, indicating 100% order ID coverage.

Two major data quality anomalies were identified:

**1. Exact Duplicate Records:** The raw dataset contains 100 exact duplicate rows. For example, order IDs O00021, O00022, and O00154 appear twice with identical customer IDs, product IDs, quantities, dates, and statuses. These duplicate records can lead to inflated sales and order-line counts.

**2. Missing Customer IDs:** The raw dataset contains 173 records with missing customer IDs. This can affect customer-level analysis and make it difficult to associate those orders with specific customers.

**Conclusion:** Although the raw and reference datasets have complete order ID coverage, the raw dataset contains duplicate records and missing customer information. These issues should be addressed during data transformation before using the data for reliable business analytics.

## 9. Save your work

Save the notebook to Drive using **File → Save a copy in Drive**. Download the SQLite database before the Colab runtime ends.

In [43]:
conn.commit()
# Run this when you have finished using the database connection.
# conn.close()

from google.colab import files
files.download('/content/retail_lab.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Student checklist

- [ ] Inspected all four CSV files.
- [ ] Loaded three raw tables into SQLite.
- [ ] Documented actual schemas and table grain.
- [ ] Created and improved all three staging views using actual field names.
- [ ] Ran data-quality checks and explained decisions.
- [ ] Answered at least three business questions with SQL.
- [ ] Compared results with `orders_clean.csv`.
- [ ] Saved the notebook and downloaded `retail_lab.db`.